# Project 1 ANSC 4040 Model Tuning

This is a notebook for Project 1 that will be used for forming the initial model. The goal is to form a RandomForestClassifier to determine the AnimalId. The large limitation of this approach is that the model will NOT fucntion when new cows are added to the data set such as on a new farm. All cows in test set must be in the training set. 

In [11]:
import pandas as pd
import joblib
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

# Load Train and Validation Sets (100 cows)

In [12]:
train_path = "train_data.csv"
val_path = "val_data.csv"

# Find the training-set cow IDs using a bounded-memory scan.
animal_ids = set()
for id_chunk in pd.read_csv(
    train_path,
    usecols=["AnimalId"],
    dtype={"AnimalId": "string"},
    chunksize=100_000,
):
    animal_ids.update(id_chunk["AnimalId"].dropna().unique())

selected_animals = set(
    pd.Series(sorted(animal_ids)).sample(n=100, random_state=42).tolist()
)

# Read only model columns and rows for those 100 cows.
model_cols = [
    "AnimalId",
    "EventDate",
    "Avgmilkflow",
    "Flow30_60Session",
    "YieldFirst2Min_Session",
    "YieldSession",
    "DurationSession_sec",
    "milking",
]

def load_selected_cows(file_path):
    matching_chunks = []
    for chunk in pd.read_csv(
        file_path,
        usecols=model_cols,
        dtype={"AnimalId": "string"},
        chunksize=100_000,
    ):
        matching = chunk[chunk["AnimalId"].isin(selected_animals)]
        if not matching.empty:
            matching_chunks.append(matching)
    if not matching_chunks:
        raise ValueError(f"None of the selected cows were found in {file_path}.")
    return pd.concat(matching_chunks, ignore_index=True)

train = load_selected_cows(train_path)
val = load_selected_cows(val_path)

print(f"Selected cows: {sorted(selected_animals)}")
print(f"Training rows: {len(train):,}; validation rows: {len(val):,}")

# Split EventDate into numerical calendar features
for data in (train, val):
    event_date = pd.to_datetime(data["EventDate"])
    data["EventYear"] = event_date.dt.year
    data["EventMonth"] = event_date.dt.month
    data["EventDay"] = event_date.dt.day
    data["EventDayOfWeek"] = event_date.dt.dayofweek

Selected cows: ['-1.1716852323791606e+18', '-1.1833355166981594e+18', '-1.3332775441306954e+17', '-2.05144505093895e+18', '-2.099013255094703e+18', '-2.221464130034193e+18', '-2.25327613911556e+18', '-2.2990285093312376e+18', '-2.3422767415993856e+18', '-2.362711626198422e+18', '-2.4428249858373806e+18', '-2.49718279280027e+18', '-2.5715480822330004e+18', '-2.972840480631011e+17', '-3.1150506377499264e+18', '-3.2709915003359217e+18', '-3.455482512856752e+18', '-3.671253680800321e+18', '-3.7808412218609434e+18', '-3.788877104804816e+18', '-3.85672259016968e+18', '-4.1398554415718595e+18', '-4.225393355328677e+18', '-4.278792536301574e+18', '-4.4539072266346624e+18', '-4.482367362136449e+18', '-4.572979853174115e+18', '-4.725483000184784e+18', '-4.7479871501442355e+17', '-4.907405775221401e+18', '-5.274629919485563e+18', '-5.297614520013029e+17', '-5.398349683912395e+18', '-5.422953988785325e+18', '-5.451041495972229e+18', '-5.501116040247302e+18', '-5.556540390544463e+18', '-6.073988599

# Tuned Hyperparameters

In [13]:
# Target variable
target = "AnimalId"

# Define feature by including columns
feature_cols = ["Avgmilkflow", "EventYear", "EventMonth", "EventDay", "EventDayOfWeek", "Flow30_60Session", "YieldFirst2Min_Session", "YieldSession", "DurationSession_sec", "milking"]



X_train = train[feature_cols]
y_train = train[target]

X_val = val[feature_cols]
y_val = val[target]

rf_model = RandomForestClassifier(
        n_estimators=20,
        random_state=42,
        max_depth=5,
        min_samples_leaf=1,
        min_samples_split=2
    )

# Fit model
rf_model.fit(X_train, y_train)

RandomForestClassifier(max_depth=5, n_estimators=20, random_state=42)

In [14]:
# Evaluate on validation rows that were not used to fit the model.
y_val_pred = rf_model.predict(X_val)
val_accuracy = accuracy_score(y_val, y_val_pred)

print(f"Validation accuracy: {val_accuracy:.2%}")
print("\nClassification report:")
print(classification_report(y_val, y_val_pred, zero_division=0))

Validation accuracy: 8.49%

Classification report:
                         precision    recall  f1-score   support

-1.1716852323791606e+18       0.00      0.00      0.00       228
-1.1833355166981594e+18       0.03      0.11      0.05        28
-1.3332775441306954e+17       0.00      0.00      0.00        59
  -2.05144505093895e+18       0.06      0.28      0.10       121
 -2.099013255094703e+18       0.05      0.08      0.06       315
 -2.221464130034193e+18       0.00      0.00      0.00       244
  -2.25327613911556e+18       0.05      0.24      0.08       147
-2.2990285093312376e+18       0.40      0.15      0.22       243
-2.3422767415993856e+18       0.00      0.00      0.00        45
 -2.362711626198422e+18       0.00      0.00      0.00       233
-2.4428249858373806e+18       0.00      0.00      0.00        99
  -2.49718279280027e+18       0.00      0.00      0.00       168
-2.5715480822330004e+18       0.00      0.00      0.00       209
 -2.972840480631011e+17       0.00    

# Hyperparameter tuning (now for 100 cows)

This small grid searches a realistic set of Random Forest settings to find the best combination without making the tuning process too slow or too broad.

In [15]:
from sklearn.model_selection import GridSearchCV

# Hyperparameter tuning with GridSearchCV
# This explores a small grid of RandomForest settings and selects the combination
# with the best cross-validation accuracy.

param_grid = {
    "n_estimators": [5, 50, 100],
    "max_depth": [None, 5, 10, 15],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [1, 4, 8],
}

grid_search = GridSearchCV(
    estimator=RandomForestClassifier(random_state=42),
    param_grid=param_grid,
    scoring="accuracy",
    cv=3,
    n_jobs=-1,
    verbose=1,
)

grid_search.fit(X_train, y_train)

print(f"Best cross-validation accuracy: {grid_search.best_score_:.2%}")
print(f"Best parameters: {grid_search.best_params_}")

best_model = grid_search.best_estimator_
y_val_pred_best = best_model.predict(X_val)
best_val_accuracy = accuracy_score(y_val, y_val_pred_best)

print(f"Validation accuracy with tuned model: {best_val_accuracy:.2%}")
print("\nClassification report for tuned model:")
print(classification_report(y_val, y_val_pred_best, zero_division=0))

results_df = pd.DataFrame(grid_search.cv_results_).sort_values("mean_test_score", ascending=False)
print("\nTop GridSearchCV results:")
print(results_df[["params", "mean_test_score", "std_test_score"]].head(10).to_string(index=False))

Fitting 3 folds for each of 108 candidates, totalling 324 fits


/Users/katrinapayton/Library/Python/3.9/lib/python/site-packages/sklearn/model_selection/_split.py:805: UserWarning: The least populated class in y has only 1 members, which is less than n_splits=3.
  warnings.warn(
/Users/katrinapayton/Library/Python/3.9/lib/python/site-packages/joblib/externals/loky/process_executor.py:782: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(


Best cross-validation accuracy: 13.26%
Best parameters: {'max_depth': 15, 'min_samples_leaf': 4, 'min_samples_split': 2, 'n_estimators': 50}
Validation accuracy with tuned model: 11.63%

Classification report for tuned model:
                         precision    recall  f1-score   support

-1.1716852323791606e+18       0.29      0.36      0.32       228
-1.1833355166981594e+18       0.14      0.96      0.25        28
-1.3332775441306954e+17       0.00      0.00      0.00        59
  -2.05144505093895e+18       0.03      0.16      0.06       121
 -2.099013255094703e+18       0.01      0.00      0.00       315
 -2.221464130034193e+18       0.09      0.02      0.03       244
  -2.25327613911556e+18       0.06      0.24      0.09       147
-2.2990285093312376e+18       0.05      0.01      0.01       243
-2.3422767415993856e+18       1.00      0.29      0.45        45
 -2.362711626198422e+18       0.58      0.08      0.14       233
-2.4428249858373806e+18       0.11      0.33      0.17    

Comparison of 5 to 100 cow hyperparameters

5- {'max_depth': 5, 'min_samples_leaf': 1, 'min_samples_split': 2, 'n_estimators': 20}
100- {'max_depth': None, 'min_samples_leaf': 8, 'min_samples_split': 2, 'n_estimators': 50}

In [8]:
%pip install xgboost

Defaulting to user installation because normal site-packages is not writeable
     |████████████████████████████████| 1.9 MB 5.4 MB/s eta 0:00:01
You should consider upgrading via the '/Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.


In [10]:
brew install libomp

SyntaxError: invalid syntax (2760538136.py, line 1)